# PREDIÇÃO NO BLOCO DE IRACEMA
- Mesmo fluxo do `Marlim/1 - Predict.ipynb`, mas a entrada é o bloco inteiro: `files/patches/*.dat` (float32 cru, já em `[0, 1]`) com o `files/patch_metadata.json` ao lado
- O bloco está em `(inline, z, crossline)` = 2751 × 2000 × 1754, cortado em 43 × 32 × 28 = 38 528 tiles de 128³ com 32 voxels de borda (só o núcleo de 64³ entra na remontagem)
- A saída vai para `<BASE_PATH>/<modelo>/iracema/masks`, um `.dat` por tile com o mesmo nome da entrada, mais o `predict.json`; são ~308 GB por modelo
- Com `SKIP_DONE`, os tiles que já têm máscara inteira ficam de fora, então dá para interromper e retomar

In [1]:
import os, sys, glob, json, gc
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

sys.path.append('../Model')                 # Network/, utils/ vivem em Model/
from Network.index import ModelNetwork

In [2]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()
print(torch.__version__)              # versão do PyTorch
print(torch.cuda.is_available())      # True se detectou a GPU

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # nome da GPU

2.7.1+cu118
True
Quadro P6000


### CARREGANDO MODELO

In [3]:
BASE_PATH   = '../Model/Backup'    # pasta dos modelos: '../Model/Backup' ou '../Documents/Marcia'
IRACEMA_DIR = 'files/patches'      # o bloco inteiro, um .dat por tile
META_PATH   = 'files/patch_metadata.json'

MODELS    = ['model_5']   # [] = todos os modelos de BASE_PATH
OVERLAP   = 0.25          # sobreposição entre janelas (0 = corte seco, do jeito que o Format corta o treino)
WINDOW    = None          # None = a janela do treino (info.json); force uma tupla para testar outra, ex: (64, 64, 64)
SKIP_DONE = True          # True pula o tile que já tem máscara: uma rodada interrompida continua de onde parou

models = MODELS or sorted(os.listdir(BASE_PATH), key=lambda n: int(n.split('_')[-1]))
models

['model_5']

In [4]:
# ESCALAR OU LISTA VIRA TUPLA DE 3 EIXOS, PARA JANELA E TILE ENTRAREM NO MESMO FORMATO
def asTriple(value, default=None):
    value = default if value is None else value
    value = [value] * 3 if np.isscalar(value) else list(value)

    if len(value) != 3:
        raise ValueError(f'esperado 3 eixos, recebido {value}')

    return tuple(int(v) for v in value)


# LÊ O .DAT CRU DO IRACEMA NO SHAPE DO TILE (NÃO É FIXO EM 128³)
def getDAT(path, shape):
    data = np.fromfile(path, dtype=np.float32)

    if data.size != int(np.prod(shape)):
        raise ValueError(f'{os.path.basename(path)}: {data.size} valores, esperado {tuple(shape)}')

    return data.reshape(shape)


# SERVE OS TILES DO BLOCO UM A UM, NO SHAPE DECLARADO NO PATCH_METADATA
class PredictDataset(Dataset):
    def __init__(self, patchDir, shape):
        self.paths = sorted(glob.glob(os.path.join(os.path.abspath(patchDir), '*.dat')))
        self.shape = asTriple(shape, default=128)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        return torch.from_numpy(getDAT(self.paths[index], self.shape))


meta = json.load(open(META_PATH))
print({k: v for k, v in meta.items() if k != 'patch_files'})
len(PredictDataset(IRACEMA_DIR, meta['patch_size'])), PredictDataset(IRACEMA_DIR, meta['patch_size']).shape

{'original_shape': [2751, 2000, 1754], 'patch_size': 128, 'overlap_percent': 25.0, 'overlap_voxels': 32, 'do_overlap': True, 'stride': 64, 'grid_shape': [43, 32, 28], 'dtype': 'float32', 'total_patches': 38528, 'percentile': 99.0, 'norm_method': 'normalize', 'vmin': -67165.125, 'vmax': 68250.0, 'mean': None, 'std': None, 'crop_origin': [0, 0, 0], 'full_shape': [2751, 2000, 1754]}


(38528, (128, 128, 128))

### JANELA DE INFERÊNCIA

In [5]:
class SlidingWindow:
    BUDGET    = 128 ** 3   # voxels por lote na GPU
    MAX_BATCH = 256

    def __init__(self, window, shape, overlap=OVERLAP):
        self.tile    = asTriple(shape)
        self.window  = asTriple(window)
        self.overlap = float(overlap)

        self.shape  = tuple(max(t, w) for t, w in zip(self.tile, self.window))   # tile menor que a janela -> reflect
        self.pad    = tuple(s - t for s, t in zip(self.shape, self.tile))
        self.starts = [self.getStarts(s, w) for s, w in zip(self.shape, self.window)]

        self.positions = [(d, h, w) for d in self.starts[0] for h in self.starts[1] for w in self.starts[2]]
        self.weight    = self.getWeight(self.window)
        self.batch     = int(np.clip(self.BUDGET // int(np.prod(self.window)), 1, self.MAX_BATCH))

    # INÍCIO DE CADA JANELA NUM EIXO; A ÚLTIMA ENCOSTA NO FIM DO TILE
    def getStarts(self, size, window):
        step = max(int(round(window * (1 - self.overlap))), 1)
        return sorted(set(list(range(0, size - window + 1, step)) + [size - window]))

    # PESO 3D SEPARÁVEL, ~1 NO CENTRO DA JANELA E ~0 NA BORDA, PARA A EMENDA NÃO VIRAR DEGRAU
    @staticmethod
    def getWeight(window):
        d, h, w = [np.hanning(n + 2)[1:-1].astype(np.float32) for n in window]
        return np.maximum(d[:, None, None] * h[None, :, None] * w[None, None, :], 1e-3)

    # PROBABILIDADE DO TILE INTEIRO, SOMANDO AS JANELAS PONDERADAS PELO PESO
    def process(self, network, tile):
        volume = tile.numpy() if torch.is_tensor(tile) else np.asarray(tile)
        volume = np.pad(volume, [(0, p) for p in self.pad], mode='reflect') if any(self.pad) else volume
        volume = torch.from_numpy(np.ascontiguousarray(volume, np.float32)).to(network.device)

        weight = torch.from_numpy(self.weight).to(network.device)
        total  = torch.zeros((network.classes, *self.shape), dtype=torch.float32, device=network.device)
        norm   = torch.zeros(self.shape, dtype=torch.float32, device=network.device)
        wd, wh, ww = self.window

        with torch.no_grad():
            for start in range(0, len(self.positions), self.batch):
                chunk  = self.positions[start:start + self.batch]
                batch  = torch.stack([volume[d:d + wd, h:h + wh, w:w + ww] for d, h, w in chunk]).unsqueeze(1)
                logits = network.model(batch)
                probs  = torch.softmax(logits, dim=1) if network.multiclass else torch.sigmoid(logits)

                for i, (d, h, w) in enumerate(chunk):
                    total[:, d:d + wd, h:h + wh, w:w + ww] += probs[i].float() * weight
                    norm[d:d + wd, h:h + wh, w:w + ww]     += weight

        probs = (total / norm).cpu().numpy().astype(np.float32)
        probs = probs[0] if network.classes == 1 else probs
        return probs[..., :self.tile[0], :self.tile[1], :self.tile[2]]

    def __repr__(self):
        return (f'SlidingWindow(janela={self.window}, tile={self.tile}, overlap={self.overlap}, 'f'janelas/tile={len(self.positions)}, batch={self.batch})')


SlidingWindow(window=(32, 32, 32), shape=128)

SlidingWindow(janela=(32, 32, 32), tile=(128, 128, 128), overlap=0.25, janelas/tile=125, batch=64)

### PREDIÇÕES NO BLOCO

In [ ]:
class IracemaPredictor:
    def __init__(self, models, base=BASE_PATH, iracemaDir=IRACEMA_DIR, metaPath=META_PATH, overlap=OVERLAP, window=WINDOW, skipDone=SKIP_DONE):
        self.models     = models
        self.base       = base
        self.iracemaDir = iracemaDir
        self.metaPath   = metaPath
        self.overlap    = overlap
        self.window     = window
        self.skipDone   = skipDone
        self.meta       = json.load(open(metaPath))

    # PASTA DAS MÁSCARAS DO BLOCO
    def outputDir(self, modelName):
        return os.path.join(self.base, modelName, 'iracema', 'masks')

    # CARREGA OS PESOS E DEVOLVE A REDE COM A JANELA EM QUE ELA FOI TREINADA
    def loadNetwork(self, modelName):
        modelPath = f'{self.base}/{modelName}'
        options   = json.load(open(f'{modelPath}/info.json', encoding='utf-8')).get('model', {})

        print(f'[{modelName}] Model Options:')
        print(json.dumps(options, indent=4))

        network = ModelNetwork(**options)
        network.model.load_state_dict(torch.load(f'{modelPath}/model.pth', map_location=network.device)['model'])
        network.model.eval()
        print(f'Weights from {modelPath} loaded successfully!\n')
        return network, asTriple(self.window or options.get('img_size'), default=128)

    # TRUE QUANDO A MÁSCARA DO TILE JÁ FOI GRAVADA INTEIRA (UM ARQUIVO CORTADO NO MEIO CONTA COMO PENDENTE)
    def hasMask(self, modelName, path, shape):
        out = os.path.join(self.outputDir(modelName), os.path.basename(path))
        return os.path.exists(out) and os.path.getsize(out) == 4 * int(np.prod(shape))

    # LOADER QUE SERVE OS TILES DO BLOCO; COM SKIP_DONE, SÓ OS QUE AINDA NÃO TÊM MÁSCARA
    def getLoader(self, modelName):
        dataset = PredictDataset(self.iracemaDir, self.meta.get('patch_size'))

        if not len(dataset):
            raise FileNotFoundError(f'nenhum .dat em {self.iracemaDir}')

        if self.skipDone:
            dataset.paths = [p for p in dataset.paths if not self.hasMask(modelName, p, dataset.shape)]

        return DataLoader(dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

    # TRUE QUANDO A masks/ JÁ TEM UM TILE PARA CADA .DAT DE ENTRADA
    def isDone(self, modelName):
        total = glob.glob(f'{self.iracemaDir}/*.dat')
        shape = asTriple(self.meta.get('patch_size'), default=128)
        return len(total) > 0 and all(self.hasMask(modelName, p, shape) for p in total)

    # REGISTRA A JANELA USADA NO BLOCO, QUE O ANALYSIS LÊ PARA A TABELA
    def export(self, modelName, sliding):
        info = {
            'model': modelName,
            'block': 'iracema',
            'tiles': len(glob.glob(f'{self.iracemaDir}/*.dat')),
            'tile_shape': list(sliding.tile),
            'img_size': list(sliding.window),
            'overlap': sliding.overlap,
            'windows_per_tile': len(sliding.positions),
        }

        path = os.path.join(self.base, modelName, 'iracema', 'predict.json')
        json.dump(info, open(path, 'w', encoding='utf-8'), ensure_ascii=False, indent=4)

    # PREDIZ OS TILES PENDENTES DO BLOCO, GRAVANDO UMA MÁSCARA POR TILE NO MESMO SHAPE DA ENTRADA
    def predictBlock(self, network, window, modelName):
        loader    = self.getLoader(modelName)
        dataset   = loader.dataset
        sliding   = SlidingWindow(window, dataset.shape, overlap=self.overlap)
        outputDir = self.outputDir(modelName)
        os.makedirs(outputDir, exist_ok=True)

        print(f'[{modelName}] iracema: {sliding} — {len(dataset)} tiles pendentes')

        if len(sliding.positions) == 1:
            print(f'[{modelName}] janela == tile: uma passada por tile — OVERLAP={self.overlap} e o peso Hann não têm efeito aqui')

        for index, tile in enumerate(tqdm(loader, desc=f'{modelName} - iracema')):
            sliding.process(network, tile[0]).tofile(os.path.join(outputDir, os.path.basename(dataset.paths[index])))

        self.export(modelName, sliding)
        print(f'Predições de {modelName} - iracema salvas em {outputDir}')

    # RODA CADA MODELO DA LISTA NO BLOCO, SE ELE AINDA ESTIVER PENDENTE
    def start(self):
        for modelName in self.models:
            if self.skipDone and self.isDone(modelName):
                print(f'[{modelName}] bloco já predito — pulando')
                continue

            if not os.path.exists(f'{self.base}/{modelName}/model.pth'):
                print(f'[{modelName}] sem model.pth — pulando')
                continue

            # O IRACEMA ESTÁ EM [0, 1]: SÓ SE APLICA A MODELO TREINADO EM DADO NESSA ESCALA (scaling 'normalize' OU 'percentile' NO info.json)
            scaling = json.load(open(f'{self.base}/{modelName}/info.json', encoding='utf-8')).get('processing', {}).get('scaling', 'normalize')
            if scaling not in ('normalize', 'percentile'):
                print(f'[{modelName}] treinado com scaling={scaling!r}, fora de [0, 1] — pulando')
                continue

            network, window = self.loadNetwork(modelName)
            self.predictBlock(network, window, modelName)

            del network
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


predictor = IracemaPredictor(models)
predictor.start()

[model_5] Model Options:
{
    "network": "dbrnet",
    "img_size": [
        128,
        128,
        128
    ],
    "classes": 1,
    "channels": 1,
    "dropout": 0.1,
    "num_filters": 32,
    "lr": 0.001
}
Weights from ../Model/Backup/model_5 loaded successfully!

[model_5] iracema: SlidingWindow(janela=(128, 128, 128), tile=(128, 128, 128), overlap=0.25, janelas/tile=1, batch=1) — 38528 tiles pendentes
[model_5] janela == tile: uma passada por tile — OVERLAP=0.25 e o peso Hann não têm efeito aqui


model_5 - iracema:   3%|▎         | 1259/38528 [10:00<4:57:02,  2.09it/s]